
# Emotion Analysis with Pretrained RoBERTa

**Model:** `pysentimiento/robertuito-emotion-analysis`  
**Dataset:** goemotions_cleaned.csv

**Objective:** Evaluate a pretrained emotion classification model *without fine-tuning* using ranking-based and multi-label metrics, then fine-tune it and see the results


## 1. Imports and Setup
We begin by importing the required libraries for NLP inference, numerical computation, and data handling.


In [1]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
import numpy as np


c:\Users\xinle\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



## 2. Load Pretrained Emotion Model
We load a RoBERTa-based emotion classifier from Hugging Face.  
This model is used *as-is* (no fine-tuning) to measure zero-shot performance.


In [ ]:

tokenizer = AutoTokenizer.from_pretrained("pysentimiento/robertuito-emotion-analysis")
model = AutoModelForSequenceClassification.from_pretrained("pysentimiento/robertuito-emotion-analysis")

labels = model.config.id2label
labels


c:\Users\xinle\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\xinle\.cache\huggingface\hub\models--pysentimiento--robertuito-emotion-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



## 3. Sanity Check: Single-Text Emotion Predictions
Before evaluating on a dataset, we test the model on a few example sentences and inspect the top-3 predicted emotions.


In [ ]:

texts = [
    "FUCK!",
    "This is an amazing day",
    "My mom died. Great.",
    "This is just great."
]

for t in texts:
    inputs = tokenizer(t, return_tensors="pt")
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)

    top3 = torch.topk(probs, k=3)
    top3_probs = top3.values[0]
    top3_indices = top3.indices[0]
    top3_emotions = [labels[i.item()] for i in top3_indices]

    print(f"Text: {t}")
    for i in range(3):
        print(f"  {i+1}. {top3_emotions[i]} ({top3_probs[i].item()*100:.2f}%)")
    print()


Text: FUCK!
  1. others (40.95%)
  2. anger (20.15%)
  3. surprise (16.95%)

Text: This is an amazing day
  1. joy (93.73%)
  2. others (3.40%)
  3. surprise (1.30%)

Text: My mom died. Great.
  1. others (46.26%)
  2. sadness (32.22%)
  3. joy (14.93%)

Text: This is just great.
  1. joy (95.55%)
  2. others (2.68%)
  3. surprise (0.74%)




## 4. Load GoEmotions Dataset
We load the GoEmotions dataset and extract:
- Text data
- Multi-label emotion annotations


In [ ]:

df = pd.read_csv("goemotions_cleaned.csv")

emotion_cols = df.columns[9:]
df = df[["text"] + list(emotion_cols)]

df.head()


,text,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1



## 5. Batch Inference (Efficient Evaluation)
To improve efficiency, predictions are generated in batches rather than one text at a time.


In [ ]:

batch_size = 32
predictions = []

for i in range(0, len(df), batch_size):
    batch_texts = df["text"].iloc[i:i+batch_size].tolist()
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    predictions.extend(probs.detach().numpy())

predictions = np.array(predictions)



## 6. Ground Truth Labels
The dataset contains multi-hot encoded emotion labels.


In [ ]:

true_labels = df[emotion_cols].astype(int).values
true_labels.shape



## 7. Evaluation Metric: Top-k Accuracy (k = 3)

**Definition:**  
A prediction is counted as correct if *at least one* of the top-k predicted emotions matches any ground-truth label.

This metric is suitable for:
- Multi-label emotion classification
- Ambiguous or overlapping emotions
- Zero-shot evaluation


In [ ]:

top_k = 3
correct_count = 0

for i in range(len(true_labels)):
    topk_preds = np.argsort(predictions[i])[::-1][:top_k]
    if any(true_labels[i][idx] == 1 for idx in topk_preds):
        correct_count += 1

top3_accuracy = correct_count / len(true_labels)
print(f"Top-{top_k} accuracy: {top3_accuracy*100:.2f}%")

To do:
- find an english version of Roberta
